<div style="background-color:#5A3516; color:#F3EEE6; padding:22px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h1 style="color:#F3EEE6; margin-bottom:0;"><b>Machine Learning II — Customer Segmentation</b></h1>
<h3 style="color:#D8C0B4; margin-top:6px;">Notebook 1 — Exploratory Data Analysis and Preprocessing</h3>
<p style="color:#D8C0B4; font-size:15px; margin-top:14px;">
This notebook prepares the <code>customer_info</code> dataset for the clustering stage. The goal is to clean the data, engineer meaningful customer features, protect identifiers from distance transformations and export an unscaled dataset ready for clustering.
</p>
</div>

<div style="background-color:#5A3516; color:#F3EEE6; padding:18px; border-radius:10px; border-left:8px solid #A8B7BA;">

<h2 style="color:#F3EEE6; margin-top:0;"><b>Index</b></h2>

<p style="color:#D8C0B4;">This index follows the full workflow from raw data inspection to the final export used for clustering.</p>

<ol>
  <li><a href="#section-1-imports-and-data-loading">Imports and Data Loading</a></li>
  <li><a href="#section-1-1-data-analysis">Initial Data Analysis</a></li>
  <li><a href="#section-1-2-duplicate-rows-analysis">Duplicate Rows Analysis</a></li>
  <li><a href="#section-1-3-missing-values-analysis">Missing Values Analysis</a></li>
  <li><a href="#section-1-4-numerical-and-categorical-columns">Numerical and Categorical Columns</a></li>
  <li><a href="#section-1-5-statistical-summary">Statistical Summary</a></li>
  <li><a href="#section-2-data-cleaning">Data Cleaning</a></li>
  <li><a href="#section-2-1-invalid-future-years">Invalid Future Years</a></li>
  <li><a href="#section-2-2-negative-values">Negative Values</a></li>
  <li><a href="#section-2-3-negative-percentages">Negative Percentages</a></li>
  <li><a href="#section-2-4-data-types">Data Type Fixes</a></li>
  <li><a href="#section-2-5-missing-values-treatment">Missing Values Treatment</a></li>
  <li><a href="#section-3-aggregation-feature-engineering">Aggregation Feature Engineering</a></li>
  <li><a href="#section-4-outlier-separation-and-imputation">Consensus Outlier Separation and KNN Imputation</a></li>
  <li><a href="#section-5-transformation-feature-engineering">Transformation Feature Engineering</a></li>
  <li><a href="#section-6-outlier-diagnostics">Outlier Diagnostics</a></li>
  <li><a href="#section-7-multivariate-analysis">Multivariate Analysis</a></li>
  <li><a href="#section-8-feature-selection-and-export">Feature Selection and Final Export</a></li>
</ol>

</div>


<a id="section-1-imports-and-data-loading"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1. Imports and Data Loading</b></h2>
</div>


In [ ]:
import warnings

import pandas as pd

warnings.filterwarnings("ignore")

import utils_eda as utils

utils.set_plot_style()
pd.set_option("display.max_columns", None)


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

We start by loading the core library used in this notebook and the project utility module. Most repeated EDA and preprocessing steps are kept in <code>utils_eda.py</code>, so the notebook remains focused on decisions, outputs and interpretation.

</div>

<a id="section-1-1-data-analysis"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1.1 Initial Data Analysis</b></h2>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

After loading the dataset, we inspect its shape, column types, and sample records. This gives us a quick sense of what needs to be cleaned before we start creating features for clustering.

</div>


In [ ]:
DATA_PATH = "../datasets/customer_info.csv"
ci = pd.read_csv(DATA_PATH)

In [ ]:
ci.sort_index(inplace=True)
display(ci.head())


In [ ]:
display(ci.tail())

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
We use the `.info()` method to verify the column names, not null counts, and data types. This allows us to identify columns that require type conversion (e.g., dates or numerical counts stored as floats) and gives us an initial glimpse of the data density.

In [ ]:
display(ci.info(memory_usage="deep"))

<a id="section-1-2-duplicate-rows-analysis"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1.2 Duplicate Rows Analysis</b></h2>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
To ensure the reliability of our clusters, we check for duplicate records. Duplicate rows can artificially inflate the importance of certain customer profiles and lead to biased segmentation results.

In [ ]:
duplicate_rows = ci[ci.duplicated()]
display(duplicate_rows.head())

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
While checking for exact row duplicates is standard, it is also important to perform a more granular search for "logical duplicates". 

In this step, we specifically look for records that share the same **Customer Name** and **Birthdate**. Identifying such cases is vital because:
* **Data Overlap:** The same individual might have been registered twice under different IDs.
* **Clustering Bias:** If the same customer profile appears multiple times, it will unfairly pull a cluster center toward its specific attributes, distorting the final segmentation.

In [ ]:
ci[ci[['customer_name', 'customer_birthdate']].duplicated()]

<a id="section-1-3-missing-values-analysis"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1.3 Missing Values Analysis</b></h2>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
    
Missing data is a common challenge in customer datasets. We generate a missing value report to:
* Quantify the percentage of missing information per feature.
* Visualize the missing data using a barplot.
* Determine which variables require **KNN Imputation** and which ones might be too empty to be useful.

In [ ]:
missing_df = utils.get_missing_percent(ci)

In [ ]:
display(missing_df.sort_values("Missing_Percent", ascending=False).head(15))

In [ ]:
utils.plot_missing_percent(missing_df)


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

We identify features with a missing value percentage higher than **30%**. This threshold is critical because:
* **Information Loss:** Variables with high sparsity often lack enough signal to be reliably imputed.
* **Model Integrity:** Including features with excessive missing data can introduce significant noise into the clustering process, potentially leading to distorted segments.


In [ ]:
high_missing = missing_df[missing_df['Missing_Percent'] > 30]
display(high_missing)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The missing value inspection shows that the dataset is usable without dropping large parts of the customer base. The missing values are concentrated in a limited group of behavioural and spend variables, which supports using imputation later instead of removing customers prematurely.
</div>

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
We evaluate the number of unique values in the `loyalty_card_number` column to determine its nature.

In [ ]:
print(f"Unique Values of column loyalty_card_number: {ci['loyalty_card_number'].nunique()}")

<a id="section-1-4-numerical-and-categorical-columns"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1.4 Numerical and Categorical Columns</b></h2>
</div>


In [ ]:
numerical_cols, categorical_cols = utils.get_column_groups(ci)

print(f"Numerical columns ({len(numerical_cols)}):")
display(pd.DataFrame(numerical_cols, columns=["Numerical Columns"]))

print(f"Categorical columns ({len(categorical_cols)}):")
display(pd.DataFrame(categorical_cols, columns=["Categorical Columns"]))


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

To properly prepare our preprocessing pipeline, we categorize the variables into Numerical and Categorical types. This distinction is essential as each group requires specific cleaning and transformation strategies.

Numerical Features (21 Variables)
These columns represent quantitative data, including spending amounts, household counts, and geographic coordinates.
* **Household & Behavior:** `kids_home`, `teens_home`, `number_complaints`, `distinct_stores_visited`.
* **Spending Habits:** `lifetime_spend_` (Groceries, Electronics, Vegetables, Non alcoholic, Alcohol, Meat, Fish, Hygiene, Videogames, Petfood).
* **Metrics & Time:** `lifetime_total_distinct_products`, `percentage_of_products_bought_promotion`, `typical_hour`, `year_first_transaction`.
* **Geospatial & IDs:** `latitude`, `longitude`, `loyalty_card_number`.

Categorical Features (3 Variables)
These columns contain qualitative information or timestamps that define the customer profile.
* **Demographics:** `customer_gender`.
* **Identification:** `customer_name`.
* **Temporal:** `customer_birthdate` (to be used for age calculation).

</div>

<a id="section-1-4-1-categorical-columns"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1.4.1 Findings in Categorical Columns</b></h2>
</div>


In [ ]:
ci[["customer_name"]]

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

A detailed inspection of the `customer_name` field shows that several names include academic prefixes such as **BSc.**, **MSc.**, and **PhD.**

* **Observation:** The dataset contains 33,038 unique customer names.
* **Feature Engineering Opportunity:** These prefixes can be used as a proxy for the customer's **education level**. They are converted into approximate years of education: 12 for customers without a prefix, 15 for BSc, 17 for MSc, and 22 for PhD.
* **Data Cleaning:** After extracting this information, the raw name field is no longer needed for modelling.

</div>


<a id="section-1-5-statistical-summary"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1.5 Statistical Summary</b></h2>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
    
We perform a statistical summary of the numerical features using the `.describe()` method. This allows us to observe the central tendency, dispersion, and range of our variables, providing critical insights into the data's distribution.


In [ ]:
print("Statistical summary for numerical columns:")
ci[numerical_cols].describe().T

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

Based on the summary table, we can identify several areas that require attention during preprocessing:

* **Constant Features:** `loyalty_card_number` does not add useful behavioural information for clustering.
* **Data Entry Errors:** `percentage_of_products_bought_promotion` contains impossible values below 0, and `year_first_transaction` contains future years.
* **Spending Variance:** The spending variables show strong dispersion, suggesting heterogeneous purchasing behaviour and the presence of extreme customers.
* **Missing Data:** Several columns have incomplete values and therefore require treatment before modelling.

</div>


In [ ]:
condition = (ci['percentage_of_products_bought_promotion'] < 0.0) | (ci['percentage_of_products_bought_promotion'] > 1.0)
rows_wrong = ci[condition]
display(rows_wrong.head())

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

We perform a targeted integrity check on the `percentage_of_products_bought_promotion` feature. Since this variable represents a ratio, any value outside the range of **[0, 1]** is considered a data entry error.

* **Condition defined:** We filter for values strictly less than 0.0 or greater than 1.0.
* **Findings:** As noted in the descriptive statistics, there are records with negative values (e.g., -1.27), which are physically impossible.
* **Objective:** Identifying these specific rows allows us to treat them as "missing data" (NaN), ensuring they are later corrected by our **KNN Imputation** strategy rather than biasing our analysis with impossible values.

</div>

In [ ]:
vars_to_plot = utils.numeric_columns_for_eda(
    ci,
    exclude_cols=["loyalty_card_number", "customer_id", "customer_loyalty_flag"],
)

utils.plot_numeric_distributions(ci, vars_to_plot)


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

The histograms allow us to understand the shape of the customer base.

#### **Spending Categories**
* **Skewness:** Most spending features are skewed to the right: many customers spend relatively little, while a smaller group spends much more.
* **Outliers:** The boxplots confirm the presence of extreme customers. These cases are separated before clustering instead of being allowed to dominate the distance calculation.

#### **Temporal & Behavioral Patterns**
* **Typical Hour:** Time of day is cyclical, so we use sine/cosine transformation to preserve the proximity between late night and early morning hours.
* **Promotion Sensitivity:** The promotion percentage helps distinguish customers who buy more frequently under promotions.

</div>


In [ ]:
display(utils.get_skewness_table(ci, vars_to_plot))


In [ ]:
utils.plot_numeric_boxplots(ci, vars_to_plot)


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The distribution plots confirm that spending variables are highly skewed, with a small number of customers spending much more than the rest. This supports two later decisions: separating the most atypical customers before fitting the clustering model, and leaving scaling as a modelling choice in the clustering notebook.
</div>

<a id="section-2-data-cleaning"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2. Data Cleaning</b></h2>
</div>


In [ ]:
ci_clean = ci.copy()
print("Initial shape:", ci_clean.shape)

<a id="section-2-1-invalid-future-years"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2.1 Detecting Invalid Future Years</b></h2>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

To convert static dates into dynamic behavioral features, we first establish the current temporal baseline.

* **Objective:** Capture the current year using `pd.Timestamp.now().year`.
* **Application:** This value serves as the reference point for calculating:
    * **Customer Age:** Derived from `customer_birthdate`.
    * **Customer Tenure:** Calculated as the difference between the current year and the `year_first_transaction`.
* **Impact:** Transforming years and dates into "Age" and "Years as Customer" provides the clustering algorithm with numerical distances that represent lifecycle stages, which is far more predictive than raw date strings.

</div>

In [ ]:
current_year = pd.Timestamp.now().year
print("Current year:", current_year)

In [ ]:
invalid_years = ci_clean[ci_clean["year_first_transaction"] > current_year]
print("Invalid future years found:")
display(invalid_years["year_first_transaction"].unique())


In [ ]:
invalid_years_rows = utils.get_invalid_years(ci_clean)

if not invalid_years_rows.empty:
    print(f"Future years detected: {invalid_years_rows['year_first_transaction'].unique()}")
    display(invalid_years_rows.head())

In [ ]:
ci_clean, future_year_count = utils.set_future_years_to_missing(ci_clean, current_year)

print(f"Future transaction years set to NaN: {future_year_count}")
print(f"Records kept: {len(ci_clean)}")


<a id="section-2-2-negative-values"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2.2 Detecting Negative Values</b></h2>
</div>


In [ ]:
negative_values = ci_clean.select_dtypes(include=['number']).lt(0).any()
negative_values

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

### Negative Value Analysis

To ensure the logical consistency of our features before modeling, we scan all numerical columns for values below zero.

* **Objective:** Use `.lt(0).any()` to detect logical anomalies across numerical datatypes.
* **Findings:**
    * **`longitude`:** Contains negative values, which is **logically valid** because Lisbon is located west of the Greenwich meridian.
    * **`percentage_of_products_bought_promotion`:** Contains negative values, which is a **logical error** for a percentage metric.
* **Impact:** Identifying these anomalies prevents the inclusion of "placeholder" values (like -1 for missing data) that would otherwise distort the mean and variance of features during the clustering process.

</div>

<a id="section-2-3-negative-percentages"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2.3 Fixing Negative Percentages</b></h2>
</div>


In [ ]:
ci_clean, invalid_promo_count = utils.set_invalid_promotion_to_missing(ci_clean)

print(f"Invalid promotion percentage values: {invalid_promo_count}")
display(ci_clean["percentage_of_products_bought_promotion"].describe())


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">


</div>

<a id="section-2-4-data-types"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2.4 Fixing Data Types</b></h2>
</div>


In [ ]:
ci_clean = utils.parse_birthdate(ci_clean)

print("Invalid birthdates after parsing:", ci_clean["customer_birthdate"].isna().sum())


In [ ]:
int_columns = [
    "kids_home",
    "teens_home",
    "number_complaints",
    "distinct_stores_visited",
    "typical_hour",
    "lifetime_total_distinct_products",
    "year_first_transaction",
]

ci_clean = utils.coerce_numeric_columns(ci_clean, int_columns)


In [ ]:
display(ci_clean[['loyalty_card_number']].head())


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

To ensure mathematical operations and time series analyses function correctly, we cast raw object columns into their appropriate structural formats.

* **Objective:** Convert based on strings dates and mixed type numerical strings into usable `datetime64` and `numeric` formats.
* **Key Transformations:**
    * **Temporal Casting:** `customer_birthdate` is parsed using a specific format string (`%m/%d/%Y %I:%M %p`) to enable age calculations.
    * **Integer Enforcement:** Columns like `kids_home` and `number_complaints` are coerced to numeric types. By using `errors='coerce'`, any not numeric noise is safely converted to `NaN`.
* **Impact:** Standardizing types is critical for the "Temporal Baseline" established earlier. For instance, `loyalty_card_number` is revealed as a Boolean style float (1.0 or NaN), which can now be treated as a categorical flag for loyalty membership.

</div>

In [ ]:
ci_clean = utils.apply_cyclic_transformation(ci_clean, "typical_hour", max_val=24)

print(ci_clean[["typical_hour", "typical_hour_sin", "typical_hour_cos"]].head())


In [ ]:
utils.plot_cyclic_hour(ci_clean)


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">


Time of day data is inherently cyclical (23:00 is closer to 01:00 than it is to 13:00). To ensure our model understands this relationship, we map the linear `typical_hour` onto a 2D coordinate system.

* **Objective:** Transform the 24-hour cycle into Sine and Cosine components using a `cyclic_transformation`.
* **The Logic:** Raw numerical hours (0-23) create a "jump" or distance gap between the end of one day and the start of the next.
    * By mapping hours to a circle, **Hour 23** and **Hour 0** become spatially adjacent, preserving the temporal continuity of human behavior.
* **Impact:** The resulting scatter plot forms a perfect circle, confirming that the "Typical Hour" has been successfully projected into a format that allows clustering algorithms to calculate meaningful distances between different times of day.

</div>

In [ ]:
print("Updated data types:")
display(ci_clean.dtypes)

In [ ]:
binary_features = [
    col for col in ci_clean.columns
    if ci_clean[col].nunique() == 2
]

print("Binary features:")
display(binary_features)


<a id="section-2-5-missing-values-treatment"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2.5 Missing Values Treatment</b></h2>
</div>


In [ ]:
print("Missing values BEFORE imputation:")
display(ci_clean.isna().sum())

In [ ]:
ci_clean = utils.fill_zero_count_columns(
    ci_clean,
    columns=["kids_home", "teens_home", "number_complaints"],
)


In [ ]:
ci_clean = utils.encode_loyalty_flag(ci_clean)

print(ci_clean["customer_loyalty_flag"].value_counts())


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">


Handling null values is a critical step to prevent model bias and calculation errors. In this section, we address the sparsity discovered in several key features.

* **Objective:** Quantify missing data using `.isna().sum()` and apply specific to the domain imputation rules.
* **Treatment Logic:**
    * **Zero Imputation:** For count variables such as `kids_home`, `teens_home`, and `number_complaints`, missing values are treated as **0**, assuming the absence of a record implies a zero count.
    * **Flag Transformation:** The `loyalty_card_number` column is renamed to `customer_loyalty_flag` and converted into a binary indicator ($1$ for members, $0$ for customers without membership).
* **Impact:** By resolving the 13,106 missing loyalty entries and filling behavioral counts, we ensure the dataset is structurally complete. This prepares the "sparse" features for statistical analysis without losing significant row volume.

</div>

In [ ]:
print(f"Shape before removing semi-constants: {ci_clean.shape}")

protected_cols = [
    'customer_id',
    'loyalty_card_number',
    'customer_loyalty_flag',
    'is_male',
    'education_level'
]

ci_clean = utils.remove_semi_constant_features(
    ci_clean,
    threshold=0.99,
    exclude_cols=protected_cols
)

print(f"Shape after removing semi-constants: {ci_clean.shape}")


<a id="section-3-aggregation-feature-engineering"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>3. Aggregation Feature Engineering</b></h2>
</div>


In [ ]:
ci_clean, age_summary = utils.add_customer_age(ci_clean)

print(f"Minimum Age: {age_summary['min_age']}")
print(f"Maximum Age: {age_summary['max_age']}")
print(f"Invalid or suspicious ages: {age_summary['invalid_count']}")


In [ ]:
utils.plot_age_distribution(ci_clean)


In [ ]:
education_df = ci_clean.apply(utils.get_education_info, axis=1)

ci_clean["education_level"] = education_df[0].astype(int)
ci_clean["clean_customer_name"] = education_df[1]

print("Distribution of identified education levels:")
print(ci_clean["education_level"].value_counts().sort_index())

display(ci_clean[["customer_name", "clean_customer_name", "education_level"]].head())


In [ ]:
ci_clean = utils.encode_gender(ci_clean)
display(ci_clean[["customer_gender", "is_male"]].head())


In [ ]:
ci_clean.drop(columns=['customer_gender'], inplace=True)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

This step turns raw demographic information into variables that can be used in modelling.

* <b>Education level:</b> extracted safely from <code>customer_name</code> using regex, so customer names are not accidentally corrupted.
* <b>Gender:</b> encoded as a binary flag, making it compatible with based on distance models.
* <b>Customer age:</b> calculated from birthdate and checked for unrealistic values before imputation.

The result is a cleaner dataset with less text noise and more useful numerical signal.
* <b>Note on the education scale:</b> the year values (12 / 15 / 17 / 22) are an assumption about typical degree duration, and customers with no academic prefix are coded as 12 — i.e. “unknown” is grouped with high school. This is a documented assumption, not a measured fact.

</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
After creating the main demographic and loyalty features, we briefly inspect their distributions. These variables are useful for profiling and interpretation, even when some of them are not used directly in the clustering distance.
</div>

In [ ]:
categorical_review_cols = [
    "is_male",
    "customer_loyalty_flag",
    "education_level",
    "kids_home",
    "teens_home",
    "number_complaints",
]

categorical_review = utils.categorical_summary(ci_clean, categorical_review_cols)
display(categorical_review)

utils.plot_categorical_summary(ci_clean, categorical_review_cols)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The categorical and count distributions help verify that the engineered features are reasonable before export. Gender is kept as a binary flag, loyalty is represented separately from the raw card number, and education is treated as an approximate numeric proxy derived from name prefixes.
</div>

In [ ]:
ci_clean.describe().T

In [ ]:
ci_clean.columns

<a id="section-4-outlier-separation-and-imputation"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>4. Consensus Outlier Separation and KNN Imputation</b></h2>
</div>


In [ ]:
ci_clean.drop(columns=["customer_name", "clean_customer_name"], inplace=True, errors="ignore")


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

Before imputing missing values, the most atypical customers are separated into an outlier dataset. The rule is conservative: a customer is only kept aside when it is simultaneously flagged by three methods.

* <b>IQR:</b> detects unusually high or low values in numerical variables.
* <b>DBSCAN:</b> identifies observations located in sparse areas of the feature space.
* <b>SOM:</b> highlights customers with high reconstruction error in a SOM map.

The thresholds are kept below the 5% reference limit. These customers are not discarded; they are exported separately and later assigned to the closest final segment in the clustering notebook.

</div>


In [ ]:
ci_clean, outlier_dataset, outlier_summary = utils.separate_outliers_and_impute_regular(
    ci_clean,
    index_col="customer_id",
    outlier_method="consensus",
    iqr_k=2.0,
    dbscan_eps=1.0,
    som_percentile=95,
    max_remove_pct=5.0,
)


In [ ]:
to_int = [
    "kids_home",
    "teens_home",
    "number_complaints",
    "distinct_stores_visited",
    "lifetime_total_distinct_products",
    "year_first_transaction",
    "customer_loyalty_flag",
    "education_level",
    "is_male",
]

ci_clean = utils.cast_nullable_int(ci_clean, to_int)


In [ ]:
ci_clean.info()

<a id="section-5-transformation-feature-engineering"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>5. Transformation Feature Engineering</b></h2>
</div>


In [ ]:
ci_clean = utils.engineer_clustering_features(
    ci_clean,
    current_year,
    keep_absolute_spend=True
)

print("Columns after feature engineering:")
display(ci_clean.columns.tolist())

absolute_spend_cols = [c for c in ci_clean.columns if c.startswith("lifetime_spend_")]

print(f"Absolute lifetime spend columns kept: {len(absolute_spend_cols)}")
for c in absolute_spend_cols:
    print("-", c)


Feature engineering is applied after data cleaning and imputation. The exported dataset keeps absolute lifetime spend variables because the final scaling decision is made in the clustering notebook.


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

## Absolute spend feature set

This feature set keeps absolute lifetime spending variables as the main spending block, together with engineered demographic and behavioural features such as age, cyclic hour, education years, tenure, loyalty flag and gender.

A separate absolute spend view is created to inspect the final feature structure before export.

</div>


In [ ]:
ci_absolute_spend = utils.make_absolute_spend_view(ci_clean)

print("Absolute-spend style dataset shape:", ci_absolute_spend.shape)
print("Absolute-spend style columns:")
for col in ci_absolute_spend.columns:
    print("-", col)


In [ ]:
ci_clean.columns

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

Here we create broader customer features from the raw transaction and demographic information.

This version keeps the **absolute lifetime spending variables** as the main spending block. This is useful because a scaled clustering model can capture both customer value and purchasing intensity across categories.

The final clustering dataset uses absolute lifetime spending variables, so customer value and spending intensity are preserved across categories.

The actual scaler and final feature set are chosen in the clustering notebook.

</div>


<a id="section-6-outlier-diagnostics"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>6. Outlier Diagnostics</b></h2>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

The plots below are used as a final visual check of the main skewed variables after preprocessing. At this point, the extreme subset has already been separated, so these diagnostics help confirm whether the regular customer base is more stable before the clustering stage.

</div>


In [ ]:
spend_cols = [col for col in ci_clean.columns if col.startswith("lifetime_spend_")]
other_numeric = [
    "customer_age",
    "distinct_stores_visited",
    "lifetime_total_distinct_products",
]
cols_to_visualize = [col for col in spend_cols + other_numeric if col in ci_clean.columns]

utils.plot_outlier_diagnostics(ci_clean, cols_to_visualize)


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
Some high values remain visible in the boxplots after preprocessing. This is expected because lifetime spending variables are naturally skewed. The goal was not to remove every univariate extreme value, but to keep valid high spending customers while separating only the most atypical multivariate cases.
</div>

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
After the conservative outlier separation, the regular customer base still contains natural variation, but the most extreme observations are kept aside. This reduces the risk that a small number of atypical customers dominate the clustering distances.
</div>

<a id="section-7-multivariate-analysis"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>7. Multivariate Analysis: Feature Correlation</b></h2>
</div>


In [ ]:
cols_to_corr = utils.correlation_columns(
    ci_clean,
    exclude_cols=["loyalty_card_number"],
)

utils.cor_heatmap(ci_clean[cols_to_corr].corr())


In [ ]:
high_corrs = utils.get_high_correlations(ci_clean, threshold=0.7)

display(high_corrs)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

At this stage, we check how strongly numerical variables are related to one another. Highly redundant features can distort based on distance clustering by giving the same information more than one weight.

In the final exported dataset, `year_first_transaction` is replaced by `tenure`, and `typical_hour` is replaced by its cyclic sine/cosine representation. The household variables are reviewed later in the clustering notebook, where only one family representation is used in the distance calculation.

</div>


In [ ]:
ci_clean

<a id="section-8-feature-selection-and-export"></a>
<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>8. Feature Selection and Final Export</b></h2>
</div>


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

## Final clustering dataset

This step prepares the dataset that will be used in the clustering notebook.

The exported dataset is intentionally **unscaled**. It includes absolute lifetime spend features and engineered behavioural variables. Scaling is not applied here because the scaler is part of the clustering model design and must be fitted consistently inside the clustering notebook.

This allows us to compare alternatives such as MinMaxScaler, RobustScaler and StandardScaler before deciding the final KMeans pipeline.

</div>

In [ ]:
final_feature_check = pd.DataFrame([
    {
        "raw_feature": "year_first_transaction",
        "replacement": "tenure",
        "status": "removed from final export",
        "reason": "tenure is the customer-lifecycle variable used downstream",
    },
    {
        "raw_feature": "typical_hour",
        "replacement": "typical_hour_sin + typical_hour_cos",
        "status": "removed from final export",
        "reason": "the cyclic representation preserves time-of-day distance",
    },
])

display(final_feature_check)

high_corrs = utils.get_high_correlations(ci_clean, threshold=0.8)
display(high_corrs)

absolute_spend_cols = [c for c in ci_clean.columns if c.startswith("lifetime_spend_")]
print("Absolute spend features kept for clustering:")
for col in absolute_spend_cols:
    print("-", col)

print()
print("Final columns exported:")
print(list(ci_clean.columns))


The preprocessing notebook exports an unscaled dataset. Scaling is treated as a modelling choice in the clustering notebook, where different scalers can be compared on the same final feature set.


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

## Export final files

The final export stores the prepared datasets used in the modelling stage. Keeping the export step separate makes the transition from preprocessing to clustering clear and reproducible.

</div>

In [ ]:
if "outlier_dataset" in locals():
    outlier_dataset = utils.align_outlier_features(
        outlier_dataset,
        reference_df=ci_clean,
        current_year=current_year,
    )


The outlier dataset is aligned with the regular dataset before export, so both files contain the same engineered variables and can be used consistently in the clustering stage.


In [ ]:
utils.export_preprocessing_outputs(
    regular_df=ci_clean,
    outlier_df=outlier_dataset if "outlier_dataset" in locals() else None,
    output_dir="../datasets",
)


<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The final summary checks that preprocessing preserved the customer base, exported the expected files and produced a complete unscaled dataset for the clustering notebook.
</div>

In [ ]:
exported_files = ["info_clustering_unscaled.csv"]
if "outlier_dataset" in locals():
    exported_files.append("outlier_dataset.csv")

preprocessing_summary = utils.preprocessing_summary_table(
    raw_df=ci,
    regular_df=ci_clean,
    outlier_df=outlier_dataset if "outlier_dataset" in locals() else None,
    exported_files=exported_files,
)

display(preprocessing_summary)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

The main unscaled clustering dataset is exported as <code>info_clustering_unscaled.csv</code>. The separated extreme customer subset is also exported as <code>outlier_dataset.csv</code>.

Both exports preserve the DataFrame index, which is important because <code>customer_id</code> was moved to the index to protect it from KNN imputation and scaling.

</div>
